# SREDT: Chain-Ladder Semantic Development Triangle for Risk Scenario Validation

This notebook demonstrates the **SREDT (Semantic Risk Event Development Triangle)** pipeline — an actuarial chain-ladder method applied to semantic relevance mass for LLM risk scenario validation.

**Pipeline overview:**
1. Generate 60 European corporate risk scenarios (energy + financials sectors)
2. Retrieve GDELT news articles per scenario (falls back to synthetic templates in demo)
3. Build a 60×5 semantic relevance-density mass matrix (L0=volume, L1=sector, L2=subsector, L3=risk-type, L4=scenario)
4. Run Venter actuarial diagnostics on training rows to select chain-ladder (CV<0.3) or BF fallback (CV>0.5)
5. Project test set L3/L4 levels using per-transition CL or BF method
6. Evaluate AUROC, Brier score, and Spearman correlation vs. flat cosine and keyword frequency baselines

**Verdict logic:** `confirmed` if Spearman improvement >0.15 and no circularity; `partial` otherwise; `disconfirmed_circularity` if L0–L4 rank r>0.95.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# Not pre-installed on Colab — always install
_pip('sentence-transformers')
_pip('loguru==0.7.2')

# Pre-installed on Colab — install locally only to match Colab versions
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3',
         'matplotlib==3.10.0', 'requests==2.32.4', 'psutil')

In [ ]:
import gc
import json
import math
import os
import re
import sys
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import psutil
import requests
from loguru import logger
from scipy import stats
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt

# Logging setup for notebook
logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")
logger.info("Imports loaded.")

## Data Loading

Load the mini demo data (pre-computed SREDT results for 3 train + 3 test scenarios).
Tries GitHub first (works in Colab after deployment), falls back to local file.

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-8ea2f1-semantic-risk-evidence-development-trian/main/round-1/experiment-1/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
logger.info(f"Loaded data: {data['metadata']['method_name']}")
logger.info(f"Train scenarios in demo data: {len(data['metadata']['full_results']['train_scenario_analysis'])}")
logger.info(f"Test scenarios in demo data: {len(data['metadata']['full_results']['test_scenario_scores'])}")

## Configuration

Tunable parameters for the SREDT pipeline. Set to minimum values for a fast demo run.
Scale up `N_TRAIN` and `N_TEST` for a fuller experiment (original values: 40 train, 20 test).

In [ ]:
# ── DEMO CONFIG (scale up for full run) ──────────────────────────────────────
N_TRAIN = 8          # original: 40 (20 energy + 20 financials)
N_TEST  = 4          # original: 20 (10 energy + 10 financials)
SIM_THRESHOLD = 0.15  # cosine similarity threshold for mass computation
BUDGET_HARD = 9.0    # USD budget cap for LLM calls

# These flags skip rate-limited external calls for the demo
USE_HARDCODED_SCENARIOS = True   # skip LLM scenario generation
_gdelt_permanently_failed = True  # skip GDELT retrieval, use synthetic articles
SKIP_LLM_JUDGE = True            # skip LLM-as-judge, use median-split proxy

# OpenRouter config (only needed when USE_HARDCODED_SCENARIOS=False or SKIP_LLM_JUDGE=False)
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
OR_BASE_URL = "https://openrouter.ai/api/v1"
OR_MODEL_FREE = "meta-llama/llama-3.3-70b-instruct:free"
OR_MODEL_FALLBACK = "google/gemini-2.0-flash-001"

cumulative_cost = 0.0
logger.info(f"Config: N_TRAIN={N_TRAIN}, N_TEST={N_TEST}, SIM_THRESHOLD={SIM_THRESHOLD}")

## Fixed Ontologies

GICS sector descriptions (L1=broad, L2=subsector) and GARP risk categories (L3) used as centroid anchors for the semantic development triangle levels.

In [ ]:
# ── FIXED ONTOLOGIES ──────────────────────────────────────────────────────────
GICS_L1 = {
    "energy": "Oil Gas Consumable Fuels Integrated Energy Equipment Services Renewables",
    "financials": "Diversified Banks Commercial Banking Capital Markets Insurance",
}
GICS_L2 = {
    "energy": "Energy sector oil gas renewables utilities",
    "financials": "Financials sector banking insurance asset management",
}
GARP_L3 = {
    "Strategic Risk": "Strategic Risk business strategy competitive position",
    "Credit Risk": "Credit Risk default counterparty exposure loss",
    "Market Risk": "Market Risk price volatility interest rate currency equity",
    "Operational Risk": "Operational Risk process failure system breakdown fraud",
    "Liquidity Risk": "Liquidity Risk funding cash flow solvency",
    "Compliance Risk": "Compliance Risk regulatory legal sanction penalty",
}
GARP_CATEGORIES = list(GARP_L3.keys())

ENERGY_COMPANIES = ["Shell", "BP", "TotalEnergies", "E.ON", "RWE", "Neste"]
FINANCIAL_COMPANIES = ["BNP Paribas", "Deutsche Bank", "ING Group", "Allianz", "AXA", "Barclays"]

# Training: 20 windows each sector in 2023; Test: 10 windows each sector in 2024 Q1-Q2
def _date_windows(start: str, n: int, step_days: int = 18) -> list:
    from datetime import date, timedelta
    d = date.fromisoformat(start)
    windows = []
    for _ in range(n):
        end = d + timedelta(days=90)
        windows.append((d.isoformat(), end.isoformat()))
        d += timedelta(days=step_days)
    return windows

TRAIN_DATES = _date_windows("2023-01-01", 20, 18)
TEST_DATES = _date_windows("2024-01-01", 10, 18)

## Step 1: Scenario Generation

Generate European corporate risk scenarios. In the demo, `USE_HARDCODED_SCENARIOS=True` bypasses the LLM call and uses template-based scenarios directly.

The full pipeline would call an LLM (via OpenRouter) to generate realistic 90-day forward-looking scenarios referencing actual market events. The `N_TRAIN` and `N_TEST` config variables control how many scenarios are used.

In [ ]:
# ── HELPER: EXTRACT JSON ARRAY ────────────────────────────────────────────────
def extract_json_array(text: str) -> str:
    text = text.strip()
    try:
        obj = json.loads(text)
        if isinstance(obj, list):
            return json.dumps(obj)
    except json.JSONDecodeError:
        pass
    text = re.sub(r"^```(?:json)?", "", text, flags=re.MULTILINE).strip()
    text = re.sub(r"```$", "", text, flags=re.MULTILINE).strip()
    m = re.search(r"(\[.*\])", text, re.DOTALL)
    if m:
        return m.group(1)
    return text


# ── HARDCODED SCENARIO TEMPLATES (FALLBACK) ────────────────────────────────────
def _hardcoded_scenarios() -> list:
    """Fallback scenario set if LLM generation fails."""
    scenarios = []
    sid = 0
    for sector, companies, dates, split in [
        ("energy", ENERGY_COMPANIES, TRAIN_DATES, "train"),
        ("energy", ENERGY_COMPANIES, TEST_DATES, "test"),
        ("financials", FINANCIAL_COMPANIES, TRAIN_DATES, "train"),
        ("financials", FINANCIAL_COMPANIES, TEST_DATES, "test"),
    ]:
        count = 20 if split == "train" else 10
        risk_cycle = GARP_CATEGORIES * (count // len(GARP_CATEGORIES) + 1)
        for i in range(count):
            company = companies[i % len(companies)]
            risk_type = risk_cycle[i]
            start, end = dates[i % len(dates)]
            text = (
                f"{company} faces {risk_type.lower()} challenges in the {sector} sector "
                f"during {start} to {end}, driven by macroeconomic headwinds and regulatory changes."
            )
            scenarios.append({
                "id": f"scen_{sid:03d}",
                "company": company,
                "sector": sector,
                "risk_type": risk_type,
                "start_date": start,
                "end_date": end,
                "text": text,
                "split": split,
            })
            sid += 1
    return scenarios


# ── STEP 1: Generate scenarios ──────────────────────────────────────────────
logger.info("Step 1: Generating scenarios...")
all_scenarios = _hardcoded_scenarios()

# Truncate to demo config sizes (N_TRAIN from each split of train, N_TEST from test)
train_all = [s for s in all_scenarios if s["split"] == "train"]
test_all  = [s for s in all_scenarios if s["split"] == "test"]
train_scenarios = train_all[:N_TRAIN]
test_scenarios  = test_all[:N_TEST]
scenarios = train_scenarios + test_scenarios

logger.info(f"Train: {len(train_scenarios)}, Test: {len(test_scenarios)}")
pd.DataFrame(scenarios)[['id','company','sector','risk_type','split']].head(6)

## Step 2: GDELT Article Retrieval

For each scenario, retrieve relevant news articles from GDELT. The pipeline has three-tier fallback:
1. `gdeltdoc` library query by company name
2. Direct GDELT v2 REST API
3. Locally-generated synthetic article titles (templates)

In this demo, `_gdelt_permanently_failed=True` skips tiers 1 and 2 and goes straight to synthetic templates.

In [ ]:
def _gdelt_direct_api(company: str, start: str, end: str, max_records: int = 100) -> list:
    """Direct GDELT v2 REST API fallback with retry on 429."""
    start_dt = start.replace("-", "") + "000000"
    end_dt = end.replace("-", "") + "235959"
    url = (
        f"https://api.gdeltproject.org/api/v2/doc/doc"
        f"?query={requests.utils.quote(company)}"
        f"&mode=artlist&maxrecords={max_records}"
        f"&startdatetime={start_dt}&enddatetime={end_dt}&format=json"
    )
    for attempt in range(3):
        try:
            resp = requests.get(url, timeout=25)
            if resp.status_code == 429:
                wait = 2 ** attempt * 3
                time.sleep(wait)
                continue
            if resp.status_code != 200:
                return []
            data = resp.json()
            articles = data.get("articles", []) or []
            return [{"title": a.get("title", ""), "url": a.get("url", "")} for a in articles if a.get("title")]
        except Exception as e:
            logger.debug(f"Direct GDELT API failed for {company} (attempt {attempt}): {e}")
            time.sleep(2)
    return []


def _generate_synthetic_articles_local(scenario: dict, n: int = 20) -> list:
    """Generate synthetic article titles locally without LLM calls."""
    company = scenario["company"]
    risk_type = scenario["risk_type"]
    sector = scenario["sector"]
    start = scenario["start_date"]

    templates = [
        f"{company} faces {risk_type.lower()} challenges amid market turbulence",
        f"{company} reports quarterly earnings under {risk_type} pressures",
        f"Analysts warn of {risk_type.lower()} exposure at {company}",
        f"{company} CEO addresses {risk_type} concerns in annual report",
        f"European regulators scrutinize {company} {risk_type.lower()} practices",
        f"{company} shares fall as {risk_type.lower()} materializes in {sector} sector",
        f"Industry watchdog flags {company} for {risk_type.lower()} non-compliance",
        f"{company} restructures operations in response to {risk_type.lower()} pressures",
        f"Credit rating agency reviews {company} amid {risk_type.lower()} fears",
        f"{company} Q1 results impacted by {risk_type.lower()} headwinds in {start[:4]}",
        f"Investors dump {company} stock after {risk_type} warning",
        f"{company} and peers face synchronized {risk_type.lower()} in {sector}",
        f"ECB warns {sector} firms including {company} on {risk_type.lower()} exposure",
        f"{company} hires ex-regulator to oversee {risk_type.lower()} compliance",
        f"EU stress test reveals {company} {risk_type.lower()} vulnerability",
        f"Hedge funds short {company} citing {risk_type.lower()} risk",
        f"{company} board convenes emergency session on {risk_type.lower()} crisis",
        f"Bond market signals distress at {company} following {risk_type} disclosure",
        f"{company} CFO resigns amid {risk_type.lower()} probe",
        f"{company} sets aside provisions for {risk_type.lower()} losses",
    ]
    return [{"title": t, "url": ""} for t in templates[:n]]


def retrieve_articles_for_scenario(scenario: dict) -> list:
    articles = []

    # Skip GDELT entirely if it's been permanently failing
    if not _gdelt_permanently_failed:
        # Primary: gdeltdoc library
        try:
            from gdeltdoc import GdeltDoc, Filters
            gd = GdeltDoc()
            f = Filters(keyword=f'"{scenario["company"]}"', start_date=scenario["start_date"], end_date=scenario["end_date"])
            df = gd.article_search(f)
            if df is not None and len(df) > 0:
                for _, row in df.iterrows():
                    title = str(row.get("title", "")).strip()
                    if title:
                        articles.append({"title": title, "url": str(row.get("url", ""))})
            time.sleep(0.5)
        except Exception as e:
            logger.debug(f"gdeltdoc failed for {scenario['company']}: {str(e)[:100]}")

        # Fallback B: direct REST API (only if not permanently failed)
        if len(articles) < 5:
            direct = _gdelt_direct_api(scenario["company"], scenario["start_date"], scenario["end_date"])
            for a in direct:
                if not any(x["title"] == a["title"] for x in articles):
                    articles.append(a)

    # Fallback C: synthetic articles (local templates, fast)
    if len(articles) < 5:
        synthetic = _generate_synthetic_articles_local(scenario, n=20)
        for a in synthetic:
            if not any(x["title"] == a["title"] for x in articles):
                articles.append(a)

    return articles[:250]


# ── STEP 2: GDELT retrieval ─────────────────────────────────────────────────
logger.info("Step 2: Retrieving articles...")
articles_dict = {}
all_corpus = []

for i, scen in enumerate(scenarios):
    arts = retrieve_articles_for_scenario(scen)
    articles_dict[scen["id"]] = arts
    all_corpus.extend(a["title"] for a in arts)

logger.info(f"Retrieved articles for {len(scenarios)} scenarios. Corpus size: {len(all_corpus)} titles.")
# Show sample articles for first scenario
print("\nSample articles for", scenarios[0]['id'], ":", scenarios[0]['company'])
for a in articles_dict[scenarios[0]['id']][:3]:
    print(" -", a['title'])

## Step 3: Embedding Setup

Load `sentence-transformers/all-MiniLM-L6-v2` for semantic embeddings. If unavailable, falls back to TF-IDF + TruncatedSVD (128 dims).

Centroid embeddings are pre-computed for each ontology level:
- **L1**: GICS broad sector description per sector
- **L2**: GICS subsector description per sector  
- **L3**: GARP risk category description per risk type

In [ ]:
# ── STEP 3: Load embedding model ────────────────────────────────────────────
def load_embedding_model():
    try:
        from sentence_transformers import SentenceTransformer
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        logger.info("Loaded sentence-transformers/all-MiniLM-L6-v2")
        return model, "sentence_transformers"
    except Exception as e:
        logger.warning(f"sentence-transformers failed: {e}, falling back to TF-IDF")
        return None, "tfidf"

def embed_texts_st(model, texts: list) -> np.ndarray:
    """Embed using sentence-transformers, normalized."""
    return model.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)

def cosine_sim_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """Cosine similarity between rows of a and single vector b. Both normalized."""
    return a @ b


class TFIDFEmbedder:
    """TF-IDF + SVD as embedding fallback when sentence-transformers unavailable."""

    def __init__(self):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        self.vec = TfidfVectorizer(max_features=5000)
        self.svd = TruncatedSVD(n_components=128, random_state=42)
        self._fitted = False
        self._corpus = []

    def fit(self, corpus: list):
        self._corpus = corpus
        X = self.vec.fit_transform(corpus)
        self.svd.fit(X)
        self._fitted = True

    def encode_normalized(self, texts: list) -> np.ndarray:
        if not self._fitted:
            self.fit(texts)
        X = self.vec.transform(texts)
        embs = self.svd.transform(X).astype(np.float32)
        norms = np.linalg.norm(embs, axis=1, keepdims=True)
        norms = np.where(norms < 1e-10, 1.0, norms)
        return embs / norms


_tfidf_embedder = None

def embed_texts_tfidf(model, texts: list) -> np.ndarray:
    global _tfidf_embedder
    if _tfidf_embedder is None:
        _tfidf_embedder = TFIDFEmbedder()
    return _tfidf_embedder.encode_normalized(texts)

def embed_texts_dispatch(model, texts: list) -> np.ndarray:
    if model is not None:
        return embed_texts_st(model, texts)
    return embed_texts_tfidf(None, texts)


st_model, embed_type = load_embedding_model()
if embed_type == "tfidf" and all_corpus:
    _tfidf_embedder = TFIDFEmbedder()
    _tfidf_embedder.fit(all_corpus)
    logger.info("TF-IDF embedder fitted on corpus")

embed_fn = embed_texts_dispatch

# Build centroid embeddings for L1, L2, L3 levels
logger.info("Building centroid embeddings...")
l1_centroids = {
    sec: embed_fn(st_model, [GICS_L1[sec]])[0]
    for sec in ["energy", "financials"]
}
l2_centroids = {
    sec: embed_fn(st_model, [GICS_L2[sec]])[0]
    for sec in ["energy", "financials"]
}
l3_centroids = {
    rt: embed_fn(st_model, [GARP_L3[rt]])[0]
    for rt in GARP_CATEGORIES
}
logger.info(f"Centroids built. Embedding type: {embed_type}")

## Step 4: SREDT Matrix Construction

Build the 60×5 semantic relevance-density mass matrix. Each cell `C(i,j)` is:

```
C(i,j) = mean_cosine_sim_above_threshold × log(1 + count_above_threshold)
```

- **L0**: `log(1 + n_articles)` — retrieval volume
- **L1**: sector-level similarity (GICS L1 centroid)
- **L2**: subsector similarity (GICS L2 centroid)
- **L3**: risk-category similarity (GARP centroid) — **latent** for test set
- **L4**: scenario-specific similarity — **latent** for test set

Test set L3/L4 are set to NaN here; they will be projected in Step 6.

In [ ]:
# ── STEP 4: SREDT ROW COMPUTATION ────────────────────────────────────────────
def compute_sredt_row(
    scenario: dict,
    articles: list,
    model,
    embed_fn,
    l1_centroids: dict,
    l2_centroids: dict,
    l3_centroids: dict,
):
    """Returns (C_row [L0..L4], flat_sim_score).

    For 'test' scenarios, L3 and L4 are NaN (genuinely latent).
    flat_sim: mean cosine sim across all articles vs. scenario embedding.
    """
    sector = scenario["sector"]
    risk_type = scenario["risk_type"]
    split = scenario["split"]

    n_articles = len(articles)
    C_L0 = float(np.log1p(n_articles))

    if n_articles == 0:
        nan_or_zero = np.nan if split == "test" else 0.0
        return np.array([C_L0, 0.0, 0.0, nan_or_zero, nan_or_zero]), 0.0

    article_texts = [a["title"] for a in articles]
    art_embs = embed_fn(model, article_texts)  # shape (n, d)

    def level_mass(centroid: np.ndarray) -> float:
        sims = art_embs @ centroid  # (n,)
        mask = sims > SIM_THRESHOLD
        k = int(mask.sum())
        if k == 0:
            return 0.0
        return float(sims[mask].mean()) * float(np.log1p(k))

    C_L1 = level_mass(l1_centroids[sector])
    C_L2 = level_mass(l2_centroids[sector])

    if risk_type not in l3_centroids:
        risk_type = GARP_CATEGORIES[0]
    C_L3_val = level_mass(l3_centroids[risk_type])

    # Scenario embedding for L4
    scen_emb = embed_fn(model, [scenario["text"]])[0]
    C_L4_val = level_mass(scen_emb)

    # Flat sim baseline: mean cosine sim (no threshold)
    flat_sim = float((art_embs @ scen_emb).mean())

    if split == "test":
        return np.array([C_L0, C_L1, C_L2, np.nan, np.nan]), flat_sim
    else:
        return np.array([C_L0, C_L1, C_L2, C_L3_val, C_L4_val]), flat_sim


# ── STEP 8: KEYWORD FREQUENCY BASELINE ───────────────────────────────────────
def compute_keyword_freq(scenario: dict, articles: list) -> float:
    """Count how many article titles mention scenario risk keywords."""
    risk_keywords = GARP_L3.get(scenario["risk_type"], "").lower().split()
    company_words = scenario["company"].lower().split()
    keywords = set(risk_keywords[:4] + company_words)
    if not articles or not keywords:
        return 0.0
    count = 0
    for a in articles:
        title_lower = a["title"].lower()
        if any(kw in title_lower for kw in keywords):
            count += 1
    return float(count) / max(len(articles), 1)


# ── Construct SREDT matrix ──────────────────────────────────────────────────
logger.info("Step 4: Constructing SREDT matrix...")
sredt_rows = []
flat_sim_scores = []
keyword_freq_scores = []

for i, scen in enumerate(scenarios):
    arts = articles_dict[scen["id"]]
    row, flat_sim = compute_sredt_row(
        scen, arts, st_model, embed_fn, l1_centroids, l2_centroids, l3_centroids
    )
    kw = compute_keyword_freq(scen, arts)
    sredt_rows.append(row)
    flat_sim_scores.append(flat_sim)
    keyword_freq_scores.append(kw)

sredt_all = np.array(sredt_rows)  # (N_TRAIN+N_TEST, 5)
train_mask = np.array([s["split"] == "train" for s in scenarios])
test_mask  = np.array([s["split"] == "test" for s in scenarios])

sredt_train = sredt_all[train_mask]  # (N_TRAIN, 5), no NaN
sredt_test_full = sredt_all[test_mask]  # (N_TEST, 5), L3/L4 are NaN
sredt_test_partial = sredt_test_full[:, :3]  # (N_TEST, 3)

train_sectors = [s["sector"] for s in scenarios if s["split"] == "train"]
test_sectors  = [s["sector"] for s in scenarios if s["split"] == "test"]
flat_sim_train = [flat_sim_scores[i] for i, s in enumerate(scenarios) if s["split"] == "train"]
flat_sim_test  = [flat_sim_scores[i] for i, s in enumerate(scenarios) if s["split"] == "test"]
kw_test = [keyword_freq_scores[i] for i, s in enumerate(scenarios) if s["split"] == "test"]

logger.info(f"SREDT train matrix: {sredt_train.shape}")

# Display matrix summary
df_sredt = pd.DataFrame(sredt_train, columns=["L0","L1","L2","L3","L4"])
df_sredt.describe().round(4)

## Step 5: Venter Actuarial Diagnostics

For each level transition L(j)→L(j+1), compute the **chain-ladder development factor** `f_j` and the **coefficient of variation** (CV) of individual development ratios.

| CV | Decision |
|---|---|
| < 0.3 | `chain_ladder_valid` — proportionality holds |
| 0.3–0.5 | `borderline` |
| > 0.5 | `bf_fallback` — Bornhuetter-Ferguson fallback |

A linear regression test (intercept p-value) is also computed to check whether a weighted CL or additive model better fits the data.

In [ ]:
# ── STEP 5: VENTER DIAGNOSTICS ────────────────────────────────────────────────
def venter_diagnostics(sredt_train: np.ndarray) -> list:
    results = []
    for j in range(4):
        col_j  = sredt_train[:, j]
        col_j1 = sredt_train[:, j + 1]
        valid = col_j > 1e-8
        x, y = col_j[valid], col_j1[valid]

        if len(x) < 3:
            results.append({"j": j, "j_label": f"L{j}→L{j+1}", "verdict": "insufficient_data"})
            continue

        f_j = float(y.sum() / x.sum())
        ratios = y / x
        cv = float(ratios.std() / ratios.mean()) if ratios.mean() > 0 else 999.0

        try:
            slope, intercept, r_val, p_val, _ = stats.linregress(x, y)
            n = len(x)
            y_hat = slope * x + intercept
            resid = y - y_hat
            mse = float((resid**2).sum() / max(n - 2, 1))
            denom = float(((x - x.mean()) ** 2).sum())
            se_int = float(np.sqrt(mse * (1 / n + (x.mean() ** 2) / max(denom, 1e-10))))
            t_int = float(intercept / se_int) if se_int > 0 else 0.0
            p_int = float(2 * stats.t.sf(abs(t_int), df=max(n - 2, 1)))
        except Exception as e:
            logger.debug(f"Linregress failed at j={j}: {e}")
            slope = r_val = p_int = 0.0
            intercept = 0.0

        if cv < 0.3:
            verdict = "chain_ladder_valid"
        elif cv < 0.5:
            verdict = "borderline"
        else:
            verdict = "bf_fallback"

        results.append({
            "j": j,
            "j_label": f"L{j}→L{j+1}",
            "f_j": round(f_j, 4),
            "cv": round(cv, 4),
            "slope": round(float(slope), 4),
            "intercept": round(float(intercept), 6),
            "intercept_pvalue": round(float(p_int), 4),
            "r_squared": round(float(r_val) ** 2, 4),
            "verdict": verdict,
        })

    return results


logger.info("Step 5: Venter diagnostics...")
venter_results = venter_diagnostics(sredt_train)

proportionality_count = sum(1 for v in venter_results if v.get("verdict") == "chain_ladder_valid")
bf_count = sum(1 for v in venter_results if v.get("verdict") == "bf_fallback")

pd.DataFrame(venter_results)[['j_label','f_j','cv','verdict']]

## Step 6: Chain-Ladder / Bornhuetter-Ferguson Projection

For each test scenario, project the latent L3 and L4 scores using per-transition diagnostics:

- **CL (chain_ladder_valid)**: `C_hat_L = C_prev × f_j`
- **BF (bf_fallback / borderline)**: `C_hat_L = C_L2 + E_prior × max(0, 1 − q_hat)` where `E_prior` is the sector mean from training and `q_hat` is the mean proportion of observed development

In [ ]:
# ── STEP 6: BF/CL PROJECTION ─────────────────────────────────────────────────
def project_test_scenarios(
    sredt_test_partial: np.ndarray,
    sredt_train: np.ndarray,
    venter_results: list,
    test_sectors: list,
    train_sectors: list,
):
    train_sectors_arr = np.array(train_sectors)
    projected_L3 = np.zeros(len(sredt_test_partial))
    projected_L4 = np.zeros(len(sredt_test_partial))

    for i, (row, sector) in enumerate(zip(sredt_test_partial, test_sectors)):
        C_L2 = float(row[2])
        sector_mask = train_sectors_arr == sector
        train_sector = sredt_train[sector_mask]
        if train_sector.shape[0] == 0:
            train_sector = sredt_train

        E_prior_L3 = float(train_sector[:, 3].mean())
        E_prior_L4 = float(train_sector[:, 4].mean())

        valid_l4 = train_sector[:, 4] > 1e-8
        q_hat_L2_to_L4 = (
            float((train_sector[valid_l4, 2] / train_sector[valid_l4, 4]).mean())
            if valid_l4.sum() > 0 else 0.5
        )
        valid_l3 = train_sector[:, 3] > 1e-8
        q_hat_L2_to_L3 = (
            float((train_sector[valid_l3, 2] / train_sector[valid_l3, 3]).mean())
            if valid_l3.sum() > 0 else 0.5
        )

        # Safe fallback if venter_results too short
        v2to3 = venter_results[2] if len(venter_results) > 2 else {"verdict": "bf_fallback"}
        v3to4 = venter_results[3] if len(venter_results) > 3 else {"verdict": "bf_fallback"}

        if v2to3.get("verdict") == "chain_ladder_valid":
            C_hat_L3 = C_L2 * v2to3["f_j"]
        else:
            C_hat_L3 = C_L2 + E_prior_L3 * max(0.0, 1.0 - q_hat_L2_to_L3)

        projected_L3[i] = C_hat_L3

        if v3to4.get("verdict") == "chain_ladder_valid":
            C_hat_L4 = C_hat_L3 * v3to4["f_j"]
        else:
            C_hat_L4 = C_L2 + E_prior_L4 * max(0.0, 1.0 - q_hat_L2_to_L4)

        projected_L4[i] = C_hat_L4

    return projected_L3, projected_L4


logger.info("Step 6: Projecting test scenarios...")
projected_L3, projected_L4 = project_test_scenarios(
    sredt_test_partial, sredt_train, venter_results, test_sectors, train_sectors
)
logger.info(f"projected_L4 — mean={projected_L4.mean():.4f} std={projected_L4.std():.4f}")

# Show projected scores per test scenario
df_proj = pd.DataFrame([
    {"scenario": s['id'], "company": s['company'], "risk": s['risk_type'],
     "projected_L3": round(projected_L3[i], 4), "projected_L4": round(projected_L4[i], 4)}
    for i, s in enumerate(test_scenarios)
])
df_proj

## Step 7: LLM-as-Judge Ground Truth

In the full pipeline, an LLM (Llama 3.3-70B or Gemini 2.0 Flash via OpenRouter) evaluates whether each test scenario materalized based on the retrieved news articles.

**In this demo** (`SKIP_LLM_JUDGE=True`): labels are set to `-1` (unknown) and the evaluation step uses a **median-split proxy** — scenarios above the median projected L4 score are labeled as materialized.

In [ ]:
# ── STEP 7: LLM-as-judge ground truth labeling ─────────────────────────────
logger.info("Step 7: LLM-as-judge ground truth labeling...")
if SKIP_LLM_JUDGE:
    logger.info("LLM judge skipped (rate limited) — using -1 labels, median-split proxy will activate")
    llm_labels = [-1] * len(test_scenarios)
    llm_justifications = ["llm_skipped_rate_limited"] * len(test_scenarios)
else:
    # Full LLM judge would call OpenRouter here
    # llm_labels, llm_justifications = llm_judge_materialization(test_scenarios, articles_dict, client)
    raise NotImplementedError("Set SKIP_LLM_JUDGE=True for demo")

yes_count = sum(1 for l in llm_labels if l == 1)
no_count  = sum(1 for l in llm_labels if l == 0)
err_count = sum(1 for l in llm_labels if l == -1)
logger.info(f"LLM judge: YES={yes_count}, NO={no_count}, ERROR/SKIPPED={err_count}")

## Step 8–10: Evaluation Metrics & Verdict

Compare SREDT projected L4 scores against two baselines:
- **Flat cosine**: mean cosine similarity of all articles vs. scenario text (no hierarchy, no threshold)
- **Keyword frequency**: fraction of articles mentioning risk-type keywords

Metrics: AUROC, Brier score, Spearman correlation.

**Circularity check**: L0–L4 Spearman rank correlation. If >0.95, SREDT collapses to monotonic transform of retrieval volume → `disconfirmed_circularity`.

In [ ]:
# ── STEP 9: EVALUATION METRICS ────────────────────────────────────────────────
def evaluate(
    projected_l4: np.ndarray,
    flat_sim: np.ndarray,
    keyword_freq: np.ndarray,
    labels: list,
) -> dict:
    valid = [i for i, lbl in enumerate(labels) if lbl != -1]
    n_valid = len(valid)
    logger.info(f"Evaluating on {n_valid} valid LLM judge labels")

    if n_valid < 2:
        logger.warning("Fewer than 2 valid labels — using median-split proxy")
        threshold = np.median(projected_l4)
        labels_proxy = [1 if projected_l4[i] >= threshold else 0 for i in range(len(projected_l4))]
        valid = list(range(len(labels_proxy)))
        y = np.array(labels_proxy)
        ground_truth_source = "median_split_proxy"
    else:
        y = np.array([labels[i] for i in valid])
        ground_truth_source = "llm_judge"

    # Check for degenerate label distribution
    if len(np.unique(y)) < 2:
        logger.warning("All labels same class — using median-split proxy")
        threshold = np.median(projected_l4[[i for i in valid]])
        y = np.array([1 if projected_l4[i] >= threshold else 0 for i in valid])
        ground_truth_source = "median_split_proxy"

    pred_sredt = MinMaxScaler().fit_transform(
        np.array([projected_l4[i] for i in valid]).reshape(-1, 1)
    ).flatten()
    pred_flat = MinMaxScaler().fit_transform(
        np.array([flat_sim[i] for i in valid]).reshape(-1, 1)
    ).flatten()
    pred_kw = MinMaxScaler().fit_transform(
        np.array([keyword_freq[i] for i in valid]).reshape(-1, 1)
    ).flatten()

    def safe_auroc(preds: np.ndarray):
        try:
            return float(roc_auc_score(y, preds))
        except Exception:
            return None

    def safe_spearman(preds: np.ndarray):
        if len(y) < 3:
            return None
        try:
            return float(spearmanr(preds, y).statistic)
        except Exception:
            return None

    return {
        "n_valid_labels": n_valid,
        "ground_truth_source": ground_truth_source,
        "sredt_auroc": safe_auroc(pred_sredt),
        "flat_auroc": safe_auroc(pred_flat),
        "keyword_auroc": safe_auroc(pred_kw),
        "sredt_brier": float(((pred_sredt - y) ** 2).mean()),
        "flat_brier": float(((pred_flat - y) ** 2).mean()),
        "keyword_brier": float(((pred_kw - y) ** 2).mean()),
        "sredt_spearman": safe_spearman(pred_sredt),
        "flat_spearman": safe_spearman(pred_flat),
        "keyword_spearman": safe_spearman(pred_kw),
    }


logger.info("Step 8: Computing evaluation metrics...")
flat_sim_arr = np.array(flat_sim_test)
kw_arr = np.array(kw_test)
eval_metrics = evaluate(projected_L4, flat_sim_arr, kw_arr, llm_labels)

# ── STEP 9: L0–L4 rank correlation (circularity check) ─────────────────
logger.info("Step 9: L0–L4 rank correlation (circularity check)...")
C_test_L0 = sredt_test_full[:, 0]
try:
    l0_l4_rank_corr = float(spearmanr(projected_L4, C_test_L0).statistic)
except Exception:
    l0_l4_rank_corr = None
logger.info(f"L0–L4 rank correlation: {l0_l4_rank_corr}")

# ── STEP 10: Determine verdict ──────────────────────────────────────────
sredt_sp = eval_metrics.get("sredt_spearman")
flat_sp  = eval_metrics.get("flat_spearman")
spearman_improvement = (
    (sredt_sp - flat_sp) if (sredt_sp is not None and flat_sp is not None) else None
)

circularity = l0_l4_rank_corr is not None and l0_l4_rank_corr > 0.95
if circularity:
    verdict = "disconfirmed_circularity"
    main_finding = (
        f"SREDT collapses to monotonic transform of article volume (L0-L4 rank r={l0_l4_rank_corr:.3f}>0.95). "
        "Semantic hierarchy adds no independent signal."
    )
elif proportionality_count == 0 and bf_count == len(venter_results):
    verdict = "disconfirmed_proportionality"
    main_finding = (
        f"All {len(venter_results)} level transitions have CV>0.5; chain-ladder assumption violated. "
        "BF fallback used throughout; SREDT provides no improvement over flat cosine."
    )
elif (
    proportionality_count >= 1
    and spearman_improvement is not None
    and spearman_improvement > 0.15
    and (l0_l4_rank_corr is None or l0_l4_rank_corr < 0.80)
):
    verdict = "confirmed"
    main_finding = (
        f"SREDT confirmed: {proportionality_count} valid chain-ladder transitions, "
        f"Spearman improvement over flat baseline = {spearman_improvement:.3f} > 0.15, "
        f"L0-L4 rank r={l0_l4_rank_corr} < 0.80 (no circularity)."
    )
else:
    verdict = "partial"
    main_finding = (
        f"Partial: {proportionality_count} valid transitions, "
        f"Spearman improvement = {spearman_improvement}, "
        f"L0-L4 rank r = {l0_l4_rank_corr}. "
        "SREDT shows some signal but does not meet all success criteria."
    )

logger.info(f"Verdict: {verdict}")
logger.info(f"Main finding: {main_finding}")

## Results & Visualization

Summary of the SREDT pipeline results: Venter diagnostics table, metric comparison, projected L4 scores vs. baselines, and the final verdict.

In [ ]:
# ── RESULTS SUMMARY ───────────────────────────────────────────────────────────
print("=" * 60)
print(f"SREDT PIPELINE RESULTS  ({N_TRAIN} train / {N_TEST} test scenarios)")
print("=" * 60)

# Venter diagnostics table
print("\n[Venter Diagnostics]")
df_venter = pd.DataFrame(venter_results)[['j_label','f_j','cv','verdict']]
print(df_venter.to_string(index=False))

# Metrics table
print("\n[Evaluation Metrics]")
metrics_rows = [
    {"Method": "SREDT",    "AUROC": eval_metrics['sredt_auroc'],   "Brier": eval_metrics['sredt_brier'],   "Spearman": eval_metrics['sredt_spearman']},
    {"Method": "Flat Cos", "AUROC": eval_metrics['flat_auroc'],    "Brier": eval_metrics['flat_brier'],    "Spearman": eval_metrics['flat_spearman']},
    {"Method": "Keyword",  "AUROC": eval_metrics['keyword_auroc'], "Brier": eval_metrics['keyword_brier'], "Spearman": eval_metrics['keyword_spearman']},
]
print(pd.DataFrame(metrics_rows).to_string(index=False))
print(f"\nGround truth source: {eval_metrics['ground_truth_source']}")
print(f"L0-L4 rank correlation: {l0_l4_rank_corr}")
print(f"Spearman improvement: {spearman_improvement}")
print(f"\nVERDICT: {verdict}")
print(f"FINDING: {main_finding}")

# ── PLOTS ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: SREDT level means for train set
ax = axes[0]
level_means = [sredt_train[:, j].mean() for j in range(5)]
level_labels = ['L0\n(volume)', 'L1\n(sector)', 'L2\n(subsector)', 'L3\n(risk-type)', 'L4\n(scenario)']
ax.bar(level_labels, level_means, color=['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336'], alpha=0.8)
ax.set_title('Mean SREDT Level Values (Train Set)')
ax.set_ylabel('Mean Relevance Mass')
ax.grid(axis='y', alpha=0.3)

# Plot 2: Venter CV by transition
ax = axes[1]
v_labels = [v['j_label'] for v in venter_results if 'cv' in v]
v_cvs    = [v['cv'] for v in venter_results if 'cv' in v]
colors   = ['green' if c < 0.3 else ('orange' if c < 0.5 else 'red') for c in v_cvs]
ax.bar(v_labels, v_cvs, color=colors, alpha=0.8)
ax.axhline(0.3, color='green', linestyle='--', label='CL valid (<0.3)')
ax.axhline(0.5, color='red',   linestyle='--', label='BF fallback (>0.5)')
ax.set_title('Venter CV by Level Transition')
ax.set_ylabel('Coefficient of Variation')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Plot 3: Projected L4 vs Flat Cosine for test scenarios
ax = axes[2]
x_pos = range(len(test_scenarios))
ax.plot(x_pos, projected_L4, 'o-', label='SREDT L4', color='#2196F3', linewidth=2)
ax.plot(x_pos, flat_sim_arr, 's--', label='Flat Cosine', color='#FF9800', linewidth=2)
ax.plot(x_pos, kw_arr, '^:', label='Keyword Freq', color='#4CAF50', linewidth=2)
ax.set_xticks(list(x_pos))
ax.set_xticklabels([s['company'][:5] for s in test_scenarios], rotation=45, ha='right')
ax.set_title('Test Scenario Scores: SREDT vs. Baselines')
ax.set_ylabel('Score')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()